
# Modern Data Platform Blueprint

This notebook builds a compact **raw → staging → curated → analytics** data platform demo using:

- DuckDB
- dbt-duckdb
- Prefect
- YAML data contracts
- UCI Online Retail dataset


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

!pip -q install duckdb==1.1.3 dbt-duckdb==1.9.0 prefect==3.0.8 pandas==2.2.2 pyarrow==17.0.0 openpyxl==3.1.5 ucimlrepo==0.0.7 pyyaml==6.0.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7

In [23]:
data_path = "/content/drive/MyDrive/projects/modern-data-platform-blueprint"


In [24]:
import os
os.chdir(data_path)
print(os.listdir())

['.DS_Store', 'docs', 'contracts', 'scripts', 'dbt_project', 'Modern_Data_Platform_Blueprint_Colab.ipynb', 'requirements.txt', 'README.md', 'prefect_flow.py']


In [6]:
import duckdb
import pandas as pd
import prefect

print("duckdb ok")
print("pandas ok")
print("prefect ok")

duckdb ok
pandas ok
prefect ok


In [7]:
!dbt --version

Core:
  - installed: 1.10.20
  - latest:    1.11.8  - Update available!

  Your version of dbt-core is out of date!
  You can find instructions for upgrading here:
  https://docs.getdbt.com/docs/installation

Plugins:
  - duckdb: 1.9.0 - Update available!

  At least one plugin is out of date with dbt-core.
  You can find instructions for upgrading here:
  https://docs.getdbt.com/docs/installation




# Data file generation

In [ ]:

import os, json, textwrap, subprocess
from pathlib import Path

BASE = Path('/content/drive/MyDrive/projects/modern-data-platform-blueprint')
(BASE / 'contracts').mkdir(parents=True, exist_ok=True)
(BASE / 'data' / 'raw').mkdir(parents=True, exist_ok=True)
(BASE / 'data' / 'warehouse').mkdir(parents=True, exist_ok=True)
(BASE / 'dbt_project' / 'models' / 'staging').mkdir(parents=True, exist_ok=True)
(BASE / 'dbt_project' / 'models' / 'curated').mkdir(parents=True, exist_ok=True)
(BASE / 'dbt_project' / 'models' / 'analytics').mkdir(parents=True, exist_ok=True)
(BASE / 'scripts').mkdir(parents=True, exist_ok=True)
(BASE / 'exports').mkdir(parents=True, exist_ok=True)

def write(rel_path, content):
    path = BASE / rel_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n")
    print("Wrote", path)


In [ ]:
write('requirements.txt', '''duckdb==1.1.3
dbt-duckdb==1.9.0
prefect==3.0.8
pandas==2.2.2
pyarrow==17.0.0
openpyxl==3.1.5
ucimlrepo==0.0.7
pyyaml==6.0.2
''')

In [ ]:
write('contracts/transactions_contract.yaml', '''dataset: online_retail
description: Raw online retail transactions from UCI
required_columns:
  InvoiceNo: string
  StockCode: string
  Description: string
  Quantity: integer
  InvoiceDate: datetime
  UnitPrice: float
  CustomerID: float
  Country: string
rules:
  not_null_columns:
  - InvoiceNo
  - StockCode
  - Quantity
  - InvoiceDate
  - UnitPrice
  - Country
  non_negative_columns:
  - UnitPrice
  allowed_negative_columns:
  - Quantity
  unique_key_hint:
  - InvoiceNo
  - StockCode
  - InvoiceDate
''')

In [ ]:
write('dbt_project/profiles.yml', '''modern_data_platform:
  outputs:
    dev:
      type: duckdb
      path: ../data/warehouse/retail.duckdb
      threads: 4
  target: dev''')

In [ ]:
write('dbt_project/dbt_project.yml', '''name: 'modern_data_platform'
version: '1.0.0'
config-version: 2

profile: 'modern_data_platform'

model-paths: ['models']
macro-paths: ['macros']
target-path: 'target'
clean-targets:
  - 'target'
  - 'dbt_packages'

models:
  modern_data_platform:
    staging:
      +materialized: view
    curated:
      +materialized: table
    analytics:
      +materialized: table''')

In [ ]:
write('dbt_project/models/sources.yml', '''version: 2

sources:
  - name: raw
    schema: main
    tables:
      - name: raw_transactions

models:
  - name: stg_transactions
    columns:
      - name: invoice_no
        tests: [not_null]
      - name: stock_code
        tests: [not_null]
      - name: invoice_ts
        tests: [not_null]
      - name: unit_price
        tests:
          - not_null
  - name: dim_customers
    columns:
      - name: customer_key
        tests: [unique, not_null]
  - name: dim_products
    columns:
      - name: product_key
        tests: [unique, not_null]
  - name: fct_order_lines
    columns:
      - name: order_line_id
        tests: [unique, not_null]''')

In [ ]:
write('dbt_project/models/staging/stg_transactions.sql', '''with src as (

    select * from {{ source('raw', 'raw_transactions') }}

),

renamed as (

    select
        cast(InvoiceNo as varchar) as invoice_no,
        cast(StockCode as varchar) as stock_code,
        trim(cast(Description as varchar)) as product_description,
        cast(Quantity as integer) as quantity,
        cast(InvoiceDate as timestamp) as invoice_ts,
        cast(UnitPrice as double) as unit_price,
        cast(CustomerID as varchar) as customer_id,
        trim(cast(Country as varchar)) as country
    from src

),

cleaned as (

    select
        invoice_no,
        stock_code,
        nullif(product_description, '') as product_description,
        quantity,
        invoice_ts,
        unit_price,
        nullif(customer_id, '') as customer_id,
        country,
        case when lower(invoice_no) like 'c%%' or quantity < 0 then true else false end as is_cancellation,
        abs(quantity) as abs_quantity,
        abs(quantity) * unit_price as gross_line_amount
    from renamed
    where invoice_no is not null
      and stock_code is not null
      and invoice_ts is not null
      and unit_price is not null
      and country is not null

)

select * from cleaned''')

In [ ]:
write('dbt_project/models/curated/dim_customers.sql', '''with base as (
    select distinct
        coalesce(customer_id, 'UNKNOWN') as customer_key,
        country
    from {{ ref('stg_transactions') }}
)
select * from base''')

In [ ]:
write('dbt_project/models/curated/dim_products.sql', '''with base as (
    select
        stock_code as product_key,
        any_value(product_description) as product_description
    from {{ ref('stg_transactions') }}
    group by 1
)
select * from base''')

In [ ]:
write('dbt_project/models/curated/dim_dates.sql', '''with base as (
    select distinct
        cast(invoice_ts as date) as date_key,
        extract(year from invoice_ts) as year_num,
        extract(month from invoice_ts) as month_num,
        strftime(cast(invoice_ts as date), '%Y-%m') as year_month
    from {{ ref('stg_transactions') }}
)
select * from base''')

In [ ]:
write('dbt_project/models/curated/fct_order_lines.sql', '''select
    md5(concat_ws('|', invoice_no, stock_code, cast(invoice_ts as varchar), coalesce(customer_id, 'UNKNOWN'))) as order_line_id,
    invoice_no,
    stock_code as product_key,
    coalesce(customer_id, 'UNKNOWN') as customer_key,
    cast(invoice_ts as date) as date_key,
    invoice_ts,
    country,
    is_cancellation,
    quantity,
    abs_quantity,
    unit_price,
    gross_line_amount
from {{ ref('stg_transactions') }}''')

In [ ]:
write('dbt_project/models/analytics/mart_monthly_revenue.sql', '''select
    d.year_month,
    sum(case when not f.is_cancellation then f.gross_line_amount else 0 end) as revenue,
    count(distinct f.invoice_no) as invoices,
    count(*) as order_lines
from {{ ref('fct_order_lines') }} f
left join {{ ref('dim_dates') }} d
  on f.date_key = d.date_key
group by 1
order by 1''')

In [ ]:
write('dbt_project/models/analytics/mart_top_customers.sql', '''select
    customer_key,
    any_value(country) as country,
    count(distinct invoice_no) as invoice_count,
    sum(case when not is_cancellation then gross_line_amount else 0 end) as total_revenue,
    max(invoice_ts) as last_order_ts
from {{ ref('fct_order_lines') }}
where customer_key <> 'UNKNOWN'
group by 1
order by total_revenue desc
limit 25''')

In [ ]:
write('dbt_project/models/analytics/mart_country_sales.sql', '''select
    country,
    sum(case when not is_cancellation then gross_line_amount else 0 end) as total_revenue,
    count(distinct invoice_no) as invoice_count,
    count(distinct customer_key) as customer_count
from {{ ref('fct_order_lines') }}
group by 1
order by total_revenue desc''')

In [ ]:
write('dbt_project/models/analytics/ai_customer_context.sql', '''select
    customer_key,
    any_value(country) as country,
    count(distinct invoice_no) as invoice_count,
    sum(case when not is_cancellation then gross_line_amount else 0 end) as lifetime_value,
    max(invoice_ts) as last_order_ts,
    count(distinct product_key) as distinct_products_purchased
from {{ ref('fct_order_lines') }}
where customer_key <> 'UNKNOWN'
group by 1''')

In [ ]:
write('scripts/validate_contracts.py', '''from __future__ import annotations
import yaml
import pandas as pd

def _check_dtype(series: pd.Series, expected: str) -> bool:
    if expected == "string":
        return True
    if expected == "integer":
        try:
            pd.to_numeric(series.dropna(), errors="raise").astype("int64")
            return True
        except Exception:
            return False
    if expected == "float":
        try:
            pd.to_numeric(series.dropna(), errors="raise").astype("float64")
            return True
        except Exception:
            return False
    if expected == "datetime":
        try:
            pd.to_datetime(series.dropna(), errors="raise")
            return True
        except Exception:
            return False
    return True

def validate_dataframe(df: pd.DataFrame, contract_path: str) -> None:
    with open(contract_path, "r") as f:
        contract = yaml.safe_load(f)

    required = contract["required_columns"]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    for col, expected_type in required.items():
        if not _check_dtype(df[col], expected_type):
            raise TypeError(f"Column {col} failed type check for expected type {expected_type}")

    rules = contract.get("rules", {})
    for col in rules.get("not_null_columns", []):
        if df[col].isna().any():
            raise ValueError(f"Column {col} contains null values")

    for col in rules.get("non_negative_columns", []):
        numeric = pd.to_numeric(df[col], errors="coerce")
        if (numeric < 0).any():
            raise ValueError(f"Column {col} contains negative values")

    print("Contract validation passed.")''')

In [ ]:
write('scripts/load_raw_to_duckdb.py', '''from __future__ import annotations
import duckdb
import pandas as pd

def load_dataframe_to_duckdb(df: pd.DataFrame, db_path: str, table_name: str = "raw_transactions") -> None:
    con = duckdb.connect(db_path)
    con.register("tmp_df", df)
    con.execute(f"create or replace table {table_name} as select * from tmp_df")
    con.close()
    print(f"Loaded {len(df):,} rows into {db_path}:{table_name}")''')

In [ ]:
write('scripts/export_analytics.py', '''from __future__ import annotations
from pathlib import Path
import duckdb

EXPORT_TABLES = [
    "mart_monthly_revenue",
    "mart_top_customers",
    "mart_country_sales",
    "ai_customer_context",
]

def export_tables(db_path: str, export_dir: str) -> None:
    export_path = Path(export_dir)
    export_path.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect(db_path)
    for table in EXPORT_TABLES:
        df = con.execute(f"select * from {table}").df()
        outfile = export_path / f"{table}.csv"
        df.to_csv(outfile, index=False)
        print(f"Exported {outfile}")
    con.close()''')

In [ ]:
write('prefect_flow.py', '''from __future__ import annotations
import os
import subprocess
from pathlib import Path

import pandas as pd
from prefect import flow, task
from ucimlrepo import fetch_ucirepo

from scripts.validate_contracts import validate_dataframe
from scripts.load_raw_to_duckdb import load_dataframe_to_duckdb
from scripts.export_analytics import export_tables

BASE_DIR = Path(__file__).resolve().parent
DB_PATH = str(BASE_DIR / "data" / "warehouse" / "retail.duckdb")
CONTRACT_PATH = str(BASE_DIR / "contracts" / "transactions_contract.yaml")
DBT_DIR = str(BASE_DIR / "dbt_project")
EXPORT_DIR = str(BASE_DIR / "exports")

@task
def fetch_dataset() -> pd.DataFrame:
    ds = fetch_ucirepo(id=352)
    df = ds.data.original.copy()
    print(df.head())
    return df

@task
def validate_raw(df: pd.DataFrame) -> None:
    validate_dataframe(df, CONTRACT_PATH)

@task
def persist_raw_csv(df: pd.DataFrame) -> str:
    out = BASE_DIR / "data" / "raw" / "online_retail.csv"
    df.to_csv(out, index=False)
    print(f"Saved raw CSV to {out}")
    return str(out)

@task
def load_raw(df: pd.DataFrame) -> None:
    load_dataframe_to_duckdb(df, DB_PATH, "raw_transactions")

@task
def run_dbt() -> None:
    env = os.environ.copy()
    env["DBT_PROFILES_DIR"] = DBT_DIR
    subprocess.run(["dbt", "deps"], cwd=DBT_DIR, check=False, env=env)
    subprocess.run(["dbt", "build"], cwd=DBT_DIR, check=True, env=env)

@task
def export_outputs() -> None:
    export_tables(DB_PATH, EXPORT_DIR)

@flow(name="modern-data-platform-blueprint")
def retail_platform_flow() -> None:
    df = fetch_dataset()
    validate_raw(df)
    persist_raw_csv(df)
    load_raw(df)
    run_dbt()
    export_outputs()

if __name__ == "__main__":
    retail_platform_flow()''')

# Code Pipeline

In [25]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/projects/modern-data-platform-blueprint")

In [26]:

import sys, os
sys.path.append(str(BASE))
os.chdir(BASE)
print("Working directory:", BASE)


Working directory: /content/drive/MyDrive/projects/modern-data-platform-blueprint


In [27]:
import os

print(os.path.exists("/content/drive/MyDrive"))
print(os.path.exists("/content/drive/MyDrive/projects"))
print(os.path.exists("/content/drive/MyDrive/projects/modern-data-platform-blueprint"))

True
True
True


In [12]:

from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=352)
df = ds.data.original.copy()
print(df.shape)
display(df.head())


(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [28]:
import os

os.makedirs("/content/drive/MyDrive/projects/modern-data-platform-blueprint/data/warehouse", exist_ok=True)

In [29]:
print(os.listdir("/content/drive/MyDrive/projects/modern-data-platform-blueprint/data"))

['warehouse']


In [30]:
with open("/content/drive/MyDrive/projects/modern-data-platform-blueprint/data/warehouse/test.txt", "w") as f:
    f.write("test")

print("file created")

file created


In [31]:
from ucimlrepo import fetch_ucirepo
from pathlib import Path
from scripts.load_raw_to_duckdb import load_dataframe_to_duckdb

ds = fetch_ucirepo(id=352)
df = ds.data.original.copy()

db_path = Path("data/warehouse/warehouse.db")
load_dataframe_to_duckdb(df, str(db_path))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 531,283 rows into data/warehouse/warehouse.db:raw_transactions


In [36]:
import duckdb

con = duckdb.connect("data/warehouse/warehouse.db")
print(con.execute("show tables").fetchall())
print(con.execute("describe raw_transactions").fetchdf())
con.close()

[('raw_transactions',)]
   column_name column_type null   key default extra
0    InvoiceNo     VARCHAR  YES  None    None  None
1    StockCode     VARCHAR  YES  None    None  None
2  Description     VARCHAR  YES  None    None  None
3     Quantity      BIGINT  YES  None    None  None
4  InvoiceDate     VARCHAR  YES  None    None  None
5    UnitPrice      DOUBLE  YES  None    None  None
6   CustomerID      DOUBLE  YES  None    None  None
7      Country     VARCHAR  YES  None    None  None


In [33]:

os.makedirs("/content/drive/MyDrive/projects/modern-data-platform-blueprint/data/raw/", exist_ok=True)
raw_csv_path = BASE / 'data' / 'raw' / 'online_retail.csv'
df.to_csv(raw_csv_path, index=False)
print(raw_csv_path)


/content/drive/MyDrive/projects/modern-data-platform-blueprint/data/raw/online_retail.csv


In [50]:
import subprocess, os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/projects/modern-data-platform-blueprint")
env = os.environ.copy()
env["DBT_PROFILES_DIR"] = str(BASE / "dbt_project")

result = subprocess.run(
    ["dbt", "run"],
    cwd=str(BASE / "dbt_project"),
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

18:35:11  Running with dbt=1.10.20
18:35:12  Registered adapter: duckdb=1.9.0
18:35:13  Found 9 models, 10 data tests, 1 source, 456 macros
18:35:13  
18:35:13  Concurrency: 4 threads (target='dev')
18:35:13  
18:35:13  1 of 9 START sql view model main.stg_transactions .............................. [RUN]
18:35:13  1 of 9 OK created sql view model main.stg_transactions ......................... [OK in 0.33s]
18:35:13  2 of 9 START sql table model main.dim_customers ................................ [RUN]
18:35:13  3 of 9 START sql table model main.dim_dates .................................... [RUN]
18:35:13  5 of 9 START sql table model main.fct_order_lines .............................. [RUN]
18:35:13  4 of 9 START sql table model main.dim_products ................................. [RUN]
18:35:16  3 of 9 OK created sql table model main.dim_dates ............................... [OK in 2.28s]
18:35:17  4 of 9 OK created sql table model main.dim_products ............................ [OK 

In [51]:
import duckdb

con = duckdb.connect("/content/drive/MyDrive/projects/modern-data-platform-blueprint/data/warehouse/warehouse.db", read_only=True)
print(con.execute("show tables").fetchall())
con.close()

[('ai_customer_context',), ('dim_customers',), ('dim_dates',), ('dim_products',), ('fct_order_lines',), ('mart_country_sales',), ('mart_monthly_revenue',), ('mart_top_customers',), ('raw_transactions',), ('stg_transactions',)]


In [41]:
import subprocess
import os

env = os.environ.copy()
env["DBT_PROFILES_DIR"] = str(BASE / "dbt_project")

result = subprocess.run(
    ["dbt", "build"],
    cwd=str(BASE / "dbt_project"),
    env=env,
    text=True,
    capture_output=True
)

print("RETURN CODE:", result.returncode)
print(result.stdout)
print(result.stderr)

RETURN CODE: 0
16:00:51  Running with dbt=1.10.20
16:00:51  Registered adapter: duckdb=1.9.0
16:00:53  Found 9 models, 10 data tests, 1 source, 456 macros
16:00:53  
16:00:53  Concurrency: 4 threads (target='dev')
16:00:53  
16:00:54  1 of 19 START sql view model main.stg_transactions ............................. [RUN]
16:00:54  1 of 19 OK created sql view model main.stg_transactions ........................ [OK in 0.21s]
16:00:54  2 of 19 START test not_null_stg_transactions_invoice_no ........................ [RUN]
16:00:54  3 of 19 START test not_null_stg_transactions_invoice_ts ........................ [RUN]
16:00:54  4 of 19 START test not_null_stg_transactions_stock_code ........................ [RUN]
16:00:54  5 of 19 START test not_null_stg_transactions_unit_price ........................ [RUN]
16:00:54  4 of 19 PASS not_null_stg_transactions_stock_code .............................. [PASS in 0.46s]
16:00:54  2 of 19 PASS not_null_stg_transactions_invoice_no ..................

In [52]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/projects/modern-data-platform-blueprint")
db_path = str(BASE / "data" / "warehouse" / "warehouse.db")

(BASE / "exports").mkdir(parents=True, exist_ok=True)

from scripts.export_analytics import export_tables

export_tables(db_path, str(BASE / "exports"))


Exported /content/drive/MyDrive/projects/modern-data-platform-blueprint/exports/mart_monthly_revenue.csv
Exported /content/drive/MyDrive/projects/modern-data-platform-blueprint/exports/mart_top_customers.csv
Exported /content/drive/MyDrive/projects/modern-data-platform-blueprint/exports/mart_country_sales.csv
Exported /content/drive/MyDrive/projects/modern-data-platform-blueprint/exports/ai_customer_context.csv


In [53]:

import duckdb
con = duckdb.connect(db_path)

for table in ['mart_monthly_revenue', 'mart_top_customers', 'mart_country_sales', 'ai_customer_context']:
    print("\n==", table, "==")
    display(con.execute(f"select * from {table} limit 10").df())



== mart_monthly_revenue ==


,year_month,revenue,invoices,order_lines
0,2010-12,823746.140,1629,41683
1,2011-01,691364.560,1120,34350
2,2011-02,523631.890,1126,27184
3,2011-03,717639.360,1531,35915
4,2011-04,537808.621,1318,29171
5,2011-05,770536.020,1731,36292
6,2011-06,761739.900,1576,36056
7,2011-07,719221.191,1540,38716
8,2011-08,759138.380,1407,34564
9,2011-09,1058590.172,1896,49323



== mart_top_customers ==


,customer_key,country,invoice_count,total_revenue,last_order_ts
0,14646.0,Netherlands,74,280206.02,2011-12-08 12:12:00
1,18102.0,United Kingdom,60,259657.30,2011-12-09 11:50:00
2,17450.0,United Kingdom,46,194550.79,2011-12-01 13:29:00
3,16446.0,United Kingdom,2,168472.50,2011-12-09 09:15:00
4,14911.0,EIRE,201,143825.06,2011-12-08 15:54:00
5,12415.0,Australia,21,124914.53,2011-11-15 14:22:00
6,14156.0,EIRE,55,117379.63,2011-11-30 10:54:00
7,17511.0,United Kingdom,31,91062.38,2011-12-07 10:12:00
8,16029.0,United Kingdom,63,81024.84,2011-11-01 10:27:00
9,12346.0,United Kingdom,1,77183.60,2011-01-18 10:01:00



== mart_country_sales ==


,country,total_revenue,invoice_count,customer_count
0,United Kingdom,9.025222e+06,18784,3922
1,Netherlands,2.854463e+05,95,9
2,EIRE,2.834540e+05,288,4
3,Germany,2.288671e+05,457,94
4,France,2.097151e+05,392,88
5,Australia,1.385213e+05,57,9
6,Spain,6.157711e+04,90,30
7,Switzerland,5.708990e+04,54,22
8,Belgium,4.119634e+04,98,25
9,Sweden,3.837833e+04,36,8



== ai_customer_context ==


,customer_key,country,invoice_count,lifetime_value,last_order_ts,distinct_products_purchased
0,12583.0,France,15,7281.38,2011-12-07 08:07:00,115
1,13748.0,United Kingdom,5,948.25,2011-09-05 09:45:00,24
2,17809.0,United Kingdom,12,5411.91,2011-11-23 12:59:00,46
3,15311.0,United Kingdom,91,60767.90,2011-12-09 12:00:00,567
4,16098.0,United Kingdom,7,2005.63,2011-09-13 09:59:00,34
5,17420.0,United Kingdom,3,598.83,2011-10-20 14:52:00,28
6,16029.0,United Kingdom,63,81024.84,2011-11-01 10:27:00,44
7,16250.0,United Kingdom,2,389.44,2011-03-23 15:07:00,22
8,13705.0,United Kingdom,3,711.86,2011-12-02 12:32:00,25
9,13747.0,United Kingdom,1,79.60,2010-12-01 10:37:00,1


In [54]:
con.close()

In [55]:
import duckdb

con = duckdb.connect("data/warehouse/warehouse.db", read_only=True)
print(con.execute("describe mart_monthly_revenue").fetchdf())
print(con.execute("select * from mart_monthly_revenue limit 5").fetchdf())
con.close()

   column_name column_type null   key default extra
0   year_month     VARCHAR  YES  None    None  None
1      revenue      DOUBLE  YES  None    None  None
2     invoices      BIGINT  YES  None    None  None
3  order_lines      BIGINT  YES  None    None  None
  year_month     revenue  invoices  order_lines
0    2010-12  823746.140      1629        41683
1    2011-01  691364.560      1120        34350
2    2011-02  523631.890      1126        27184
3    2011-03  717639.360      1531        35915
4    2011-04  537808.621      1318        29171


In [56]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/projects/modern-data-platform-blueprint")
os.chdir(BASE)

!python scripts/visualize_analytics.py

Saved /content/drive/MyDrive/projects/modern-data-platform-blueprint/exports/figures/monthly_revenue_trend.png
Saved /content/drive/MyDrive/projects/modern-data-platform-blueprint/exports/figures/top_customers.png
Saved /content/drive/MyDrive/projects/modern-data-platform-blueprint/exports/figures/country_sales.png


In [ ]:

# Optional: run the whole thing through Prefect
#!python prefect_flow.py


In [ ]:

# Zip the repo so you can download it from Colab
#import shutil
#shutil.make_archive('/content/modern-data-platform-blueprint', 'zip', BASE)
#print('Created /content/modern-data-platform-blueprint.zip')
